# 📝 DATATHON 2026 — PHẦN 1: CÂU HỎI TRẮC NGHIỆM (ĐỀ MỚI)
**Mục tiêu:** Tính toán trực tiếp từ dữ liệu để tìm đáp án chính xác cho 10 câu hỏi trắc nghiệm.

---

In [25]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
DATA = Path('../dataset')

orders       = pd.read_csv(DATA / 'orders.csv',      parse_dates=['order_date'])
order_items  = pd.read_csv(DATA / 'order_items.csv')
products     = pd.read_csv(DATA / 'products.csv')
customers    = pd.read_csv(DATA / 'customers.csv')
returns      = pd.read_csv(DATA / 'returns.csv',     parse_dates=['return_date'])
payments     = pd.read_csv(DATA / 'payments.csv')
web_traffic  = pd.read_csv(DATA / 'web_traffic.csv', parse_dates=['date'])
geography    = pd.read_csv(DATA / 'geography.csv')
sales        = pd.read_csv(DATA / 'sales.csv',       parse_dates=['Date'])

print('✅ Đã tải xong tất cả bảng dữ liệu.')

✅ Đã tải xong tất cả bảng dữ liệu.


---
## ❓ Q1 — Trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap)
> *Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)*
> 
> A) 30 ngày | B) 90 ngày | C) 144 ngày | D) 365 ngày

In [26]:
multi_buyers = orders.groupby('customer_id').filter(lambda x: len(x) > 1)
gaps = []
for cid, grp in multi_buyers.sort_values('order_date').groupby('customer_id'):
    dates = grp['order_date'].sort_values().values
    diffs = pd.Series(dates[1:]) - pd.Series(dates[:-1])
    gaps.extend([d.days for d in diffs])

median_gap = np.median(gaps)
print(f'Số khoảng cách được tính: {len(gaps):,}')
print(f'Trung vị inter-order gap: {median_gap:.1f} ngày')
print(f'Mean inter-order gap: {np.mean(gaps):.1f} ngày')

options = [30, 90, 144, 365]
closest = min(options, key=lambda x: abs(x - median_gap))
ans_q1 = f'{closest} ngày'
print(f'\n✅ ĐÁP ÁN Q1: {ans_q1}')

Số khoảng cách được tính: 556,699
Trung vị inter-order gap: 144.0 ngày
Mean inter-order gap: 285.6 ngày

✅ ĐÁP ÁN Q1: 144 ngày


---
## ❓ Q2 — Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất
> *Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?*
>
> A) Premium | B) Performance | C) Activewear | D) Standard

In [27]:
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
margin_by_segment = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print('Tỷ suất lợi nhuận gộp trung bình theo phân khúc:')
print(margin_by_segment)
ans_q2 = margin_by_segment.idxmax()
print(f'\n✅ ĐÁP ÁN Q2: {ans_q2}')

Tỷ suất lợi nhuận gộp trung bình theo phân khúc:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64

✅ ĐÁP ÁN Q2: Standard


---
## ❓ Q3 — Lý do trả hàng phổ biến nhất trong danh mục Streetwear
> *Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?*
>
> A) defective | B) wrong_size | C) changed_mind | D) not_as_described

In [28]:
streetwear_products = products[products['category'] == 'Streetwear'][['product_id']]
streetwear_returns  = returns.merge(streetwear_products, on='product_id', how='inner')
reason_counts = streetwear_returns['return_reason'].value_counts()
print('Số lần trả hàng theo lý do trong danh mục Streetwear:')
print(reason_counts)
ans_q3 = reason_counts.idxmax()
print(f'\n✅ ĐÁP ÁN Q3: {ans_q3}')

Số lần trả hàng theo lý do trong danh mục Streetwear:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

✅ ĐÁP ÁN Q3: wrong_size


---
## ❓ Q4 — Nguồn truy cập có tỷ lệ chuyển đổi trung bình cao nhất
> *Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ chuyển đổi trung bình (conversion_rate) cao nhất trên tất cả các ngày mà nguồn đó xuất hiện?*
>
> A) organic_search | B) paid_search | C) email_campaign | D) social_media

In [29]:
# Tính số đơn hàng theo ngày và nguồn
daily_orders = orders.groupby(['order_date', 'order_source']).size().reset_index(name='orders_count')

# Kết nối web_traffic với daily_orders để có số đơn hàng
merged_traffic = web_traffic.merge(
    daily_orders, 
    left_on=['date', 'traffic_source'], 
    right_on=['order_date', 'order_source'], 
    how='left'
)
merged_traffic['orders_count'] = merged_traffic['orders_count'].fillna(0)

# Tính tỷ lệ chuyển đổi (conversion rate) = số đơn / số phiên
merged_traffic['conversion_rate'] = merged_traffic['orders_count'] / merged_traffic['sessions']

cr_by_source = merged_traffic.groupby('traffic_source')['conversion_rate'].mean().sort_values(ascending=False)
print('Tỷ lệ chuyển đổi trung bình theo nguồn truy cập:')
print(cr_by_source)
ans_q4 = cr_by_source.idxmax()
print(f'\n✅ ĐÁP ÁN Q4: {ans_q4}')

Tỷ lệ chuyển đổi trung bình theo nguồn truy cập:
traffic_source
organic_search    0.002055
paid_search       0.001624
social_media      0.001505
email_campaign    0.000867
referral          0.000711
direct            0.000585
Name: conversion_rate, dtype: float64

✅ ĐÁP ÁN Q4: organic_search


---
## ❓ Q5 — Tỷ lệ dòng order_items có áp dụng khuyến mãi
> *Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?*
>
> A) 12% | B) 25% | C) 39% | D) 54%

In [30]:
total_rows = len(order_items)
promo_rows = order_items['promo_id'].notna().sum()
promo_pct = (promo_rows / total_rows) * 100

print(f'Tổng số dòng order_items: {total_rows:,}')
print(f'Dòng có promo_id không null: {promo_rows:,}')
print(f'Tỷ lệ có khuyến mãi: {promo_pct:.2f}%')

options = [12, 25, 39, 54]
closest = min(options, key=lambda x: abs(x - promo_pct))
ans_q5 = f'{closest}%'
print(f'\n✅ ĐÁP ÁN Q5: {ans_q5}')

Tổng số dòng order_items: 714,669
Dòng có promo_id không null: 276,316
Tỷ lệ có khuyến mãi: 38.66%

✅ ĐÁP ÁN Q5: 39%


---
## ❓ Q6 — Nhóm tuổi có số đơn hàng trung bình trên khách hàng cao nhất
> *Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất?*
>
> A) 55+ | B) 25-34 | C) 35-44 | D) 45-54

In [31]:
cust_with_age = customers[customers['age_group'].notna()][['customer_id', 'age_group']]
order_counts = orders.groupby('customer_id').size().reset_index(name='order_count')
merged = cust_with_age.merge(order_counts, on='customer_id', how='left')
merged['order_count'] = merged['order_count'].fillna(0)
avg_orders_by_age = merged.groupby('age_group')['order_count'].mean().sort_values(ascending=False)

print('Số đơn hàng trung bình trên mỗi khách hàng theo nhóm tuổi:')
print(avg_orders_by_age)
ans_q6 = avg_orders_by_age.idxmax()
print(f'\n✅ ĐÁP ÁN Q6: {ans_q6}')

Số đơn hàng trung bình trên mỗi khách hàng theo nhóm tuổi:
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: order_count, dtype: float64

✅ ĐÁP ÁN Q6: 55+


---
## ❓ Q7 — Vùng tạo ra tổng doanh thu cao nhất
> *Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?*
>
> A) West | B) Central | C) East | D) Cả ba vùng có doanh thu xấp xỉ bằng nhau

*Lưu ý: sales_train.csv ám chỉ dữ liệu train (orders <= 2022-12-31). Ta tính doanh thu bằng unit_price * quantity (unit_price đã bao gồm khuyến mãi).*

In [32]:
# Tính doanh thu từ order_items (revenue = unit_price * quantity)
# Lưu ý: unit_price đã là giá sau khuyến mãi nên không trừ discount_amount
order_items['line_revenue'] = order_items['unit_price'] * order_items['quantity']

# Kết nối để lấy thông tin địa lý
oi_with_zip = order_items.merge(orders[['order_id', 'zip', 'order_date']], on='order_id', how='left')
oi_train = oi_with_zip[oi_with_zip['order_date'] <= '2022-12-31']
oi_geo = oi_train.merge(geography[['zip', 'region']], on='zip', how='left')

revenue_by_region = oi_geo.groupby('region')['line_revenue'].sum().sort_values(ascending=False)
print('Tổng doanh thu theo vùng trong tập train:')
print(revenue_by_region)

top_region = revenue_by_region.idxmax()

# Kiểm tra xem có xấp xỉ nhau không
if (revenue_by_region.max() - revenue_by_region.min()) / revenue_by_region.max() < 0.05:
    ans_q7 = 'Cả ba vùng có doanh thu xấp xỉ bằng nhau'
else:
    ans_q7 = top_region

print(f'\n✅ ĐÁP ÁN Q7: {ans_q7}')

Tổng doanh thu theo vùng trong tập train:
region
East       7.637533e+09
Central    4.941908e+09
West       3.851035e+09
Name: line_revenue, dtype: float64

✅ ĐÁP ÁN Q7: East


---
## ❓ Q8 — Phương thức thanh toán nhiều nhất trong đơn hàng bị hủy
> *Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?*
>
> A) credit_card | B) cod | C) paypal | D) bank_transfer

In [33]:
cancelled = orders[orders['order_status'] == 'cancelled']
payment_counts = cancelled['payment_method'].value_counts()
print('Phương thức thanh toán trong đơn hàng bị hủy:')
print(payment_counts)
ans_q8 = payment_counts.idxmax()
print(f'\n✅ ĐÁP ÁN Q8: {ans_q8}')

Phương thức thanh toán trong đơn hàng bị hủy:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

✅ ĐÁP ÁN Q8: credit_card


---
## ❓ Q9 — Kích thước sản phẩm có tỷ lệ trả hàng cao nhất
> *Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?*
>
> A) S | B) M | C) L | D) XL

In [34]:
oi_with_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
target_sizes = ['S', 'M', 'L', 'XL']
order_count_by_size = oi_with_size[oi_with_size['size'].isin(target_sizes)].groupby('size').size()

ret_with_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
return_count_by_size = ret_with_size[ret_with_size['size'].isin(target_sizes)].groupby('size').size()

return_rate = (return_count_by_size / order_count_by_size).sort_values(ascending=False)
print('Tỷ lệ trả hàng theo kích thước:')
for sz, rate in return_rate.items():
    print(f'Size {sz}: {return_count_by_size.get(sz, 0):,} / {order_count_by_size.get(sz, 0):,} = {rate:.4f} ({rate*100:.2f}%)')

ans_q9 = return_rate.idxmax()
print(f'\n✅ ĐÁP ÁN Q9: {ans_q9}')

Tỷ lệ trả hàng theo kích thước:
Size S: 9,723 / 172,042 = 0.0565 (5.65%)
Size L: 9,741 / 173,174 = 0.0562 (5.62%)
Size M: 9,820 / 176,428 = 0.0557 (5.57%)
Size XL: 10,655 / 193,025 = 0.0552 (5.52%)

✅ ĐÁP ÁN Q9: S


---
## ❓ Q10 — Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất
> *Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?*
>
> A) 1 kỳ (trả một lần) | B) 3 kỳ | C) 6 kỳ | D) 12 kỳ

In [35]:
avg_by_installment = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print('Giá trị thanh toán trung bình theo số kỳ trả góp:')
print(avg_by_installment)

best_installment = avg_by_installment.idxmax()

if best_installment == 1:
    ans_q10 = '1 kỳ (trả một lần)'
else:
    ans_q10 = f'{best_installment} kỳ'

print(f'\n✅ ĐÁP ÁN Q10: {ans_q10}')

Giá trị thanh toán trung bình theo số kỳ trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64

✅ ĐÁP ÁN Q10: 6 kỳ


---
## 📊 TỔNG HỢP ĐÁP ÁN PHẦN 1

In [36]:
print('=' * 60)
print('   BẢNG ĐÁP ÁN PHẦN 1 — TRẮC NGHIỆM (CẬP NHẬT THEO ĐỀ MỚI)')
print('=' * 60)
print('Câu  | Đáp án')
print('-' * 60)
print(f'Q1   | {ans_q1}')
print(f'Q2   | {ans_q2}')
print(f'Q3   | {ans_q3}')
print(f'Q4   | {ans_q4}')
print(f'Q5   | {ans_q5}')
print(f'Q6   | {ans_q6}')
print(f'Q7   | {ans_q7}')
print(f'Q8   | {ans_q8}')
print(f'Q9   | {ans_q9}')
print(f'Q10  | {ans_q10}')
print('=' * 60)

   BẢNG ĐÁP ÁN PHẦN 1 — TRẮC NGHIỆM (CẬP NHẬT THEO ĐỀ MỚI)
Câu  | Đáp án
------------------------------------------------------------
Q1   | 144 ngày
Q2   | Standard
Q3   | wrong_size
Q4   | organic_search
Q5   | 39%
Q6   | 55+
Q7   | East
Q8   | credit_card
Q9   | S
Q10  | 6 kỳ
